In [1]:
%%capture
!pip install bitsandbytes sentencepiece peft trl

In [5]:
model_name = "/kaggle/input/llama-3.1/transformers/8b/2"
max_seq_length = 2048
learning_rate = 2e-4
weight_decay = 0.01
max_steps = 120
warmup_steps = 10
batch_size = 2
gradient_accumulation_steps = 4
lr_scheduler_type = "linear"
optimizer = "adamw_8bit"
use_gradient_checkpointing = True
random_state = 3407

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

token = None
HAS_BFLOAT16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if HAS_BFLOAT16 else torch.float16
load_in_4bit = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "balanced", # for balanced_low_0 sequential auto
    torch_dtype = dtype,
    quantization_config = bnb_config if load_in_4bit else None,
    token = None,
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    model_max_length = max_seq_length,
    padding_side = "right",
    token = None,
)

model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, TaskType, get_peft_model

from transformers import set_seed as transformers_set_seed
transformers_set_seed(random_state) # Must set since LoRA weights get initialized.

lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout   = 0,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)
if load_in_4bit:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing = use_gradient_checkpointing,
    )
elif use_gradient_checkpointing:
    model.gradient_checkpointing_enable()
model = get_peft_model(model, lora_config)

# ‌SlimOrca‌
‌SlimOrca‌是一个经过优化的数据集，主要用于训练语言模型，特别是在聊天和数学推理任务中表现出色。SlimOrca是‌OpenOrca‌数据集的一个子集，通过使用GPT-4对数据进行筛选和验证，减少了错误答案的数量，从而将数据集大小缩减到约500k条记录，同时保持了与原始数据集相当的训练质量‌
1
。

特点和用途
SlimOrca的特点包括：

‌数据筛选‌：使用GPT-4对数据进行筛选，去除基于人类注释认为错误的答案，确保数据的准确性‌
1
。
‌数据规模‌：包含约500k条GPT-4完成的记录，相比原始数据集减少了计算需求，但仍能提供高质量的训练效果‌
1
。
‌应用场景‌：主要用于训练语言模型，特别是在聊天和数学推理任务中表现出色。通过QLoRA微调‌Mistral‌模型，可以获得一个能力较强的聊天模型‌
2
。

In [10]:
#@title Slim Orca data prep
from datasets import load_dataset
dataset = load_dataset("Open-Orca/SlimOrca", split = "train")
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    mapper = {"system" : "SYSTEM:", "human" : "USER:", "gpt" : "ASSISTANT:"}
    end_mapper = {"system" : "\n\n", "human" : "\n", "gpt" : "</s>\n"}
    for convo in convos:
        text = "".join(f"{mapper[(turn := x['from'])]} {x['value']}{end_mapper[turn]}" for x in convo)
        texts.append(text)
    return { "text" : texts, }
pass

dataset = dataset.map(formatting_prompts_func, batched = True,)

README.md:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

oo-labeled_correct.gpt4.sharegpt.jsonl:   0%|          | 0.00/986M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/517982 [00:00<?, ? examples/s]

Map:   0%|          | 0/517982 [00:00<?, ? examples/s]

In [12]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from transformers.utils import logging

logging.set_verbosity_info()

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps = warmup_steps,
        max_steps = max_steps,
        learning_rate = learning_rate,
        fp16 = not HAS_BFLOAT16,
        bf16 = HAS_BFLOAT16,
        logging_steps = 1,
        output_dir = "outputs",
        optim = optimizer,
        weight_decay = weight_decay,
        lr_scheduler_type = lr_scheduler_type,
        seed = random_state,
        report_to = "none",
    ),
)

PyTorch: setting up devices
loading file tokenizer.json
loading file tokenizer.model
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Applying chat template to train dataset:   0%|          | 0/517982 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/517982 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/517982 [00:00<?, ? examples/s]

You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend


In [13]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
9.137 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: conversations, text. If conversations, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 517,982
  Num Epochs = 1
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 120
  Number of trainable parameters = 41,943,040
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pas

Step,Training Loss
1,7.424200
2,8.115300
3,8.789700
4,7.891700
5,8.029600
6,7.195300
7,7.983800


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")